In [ ]:
from pixell import colorize
colorize.mpl_register("planck")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

import numpy as np
from pixell import enmap, enplot, reproject
import glob
import matplotlib.pyplot as plt
import emcee, corner
from astropy.io import fits
import sys
from astropy import units as u, constants as const
sys.path.insert(0, '../src')
#sys.path.insert(0, "/home/gill/apps/szpack/python")
import warnings
warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
import yaml
import itertools
from pixell import enmap

import bandpass as bp
import covariance as cov
import model
import utils as ut

import SZpack as SZ

from astropy.coordinates import SkyCoord
import astropy.units as u

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from astropy.visualization import ZScaleInterval
# from astropy.visualization import ImageNormalize

# # use latex for the plot
# plt.rc('text', usetex=True)
# plt.rc('font', family='sans-serif', size=25)

# # Compute ZScale limits
# # Create the plot with WCS projection
# plt.figure(figsize=(8, 8))

# flux_factor = ut.flux_factor("pa5", 98)

# emap_jy = emap / flux_factor**2

# ax = plt.subplot(projection=emap_jy.wcs)

# norm = ImageNormalize(emap_jy, interval=ZScaleInterval())

# # Display the map with ZScale normalization
# im = ax.imshow(emap_jy, origin='lower', cmap='viridis', norm=norm, interpolation='none')

# ra = ax.coords[0]
# dec = ax.coords[1]

# # Set labels
# ra.set_axislabel('Right Ascension')
# dec.set_axislabel('Declination')

# # Ensure tick labels are in float degrees
# ra.set_major_formatter('d')
# dec.set_major_formatter('d')

# ax.invert_xaxis()
# plt.colorbar(im,fraction=0.046, pad=0.04, label='(Jy/sr)$^{-2}$')

# #plt.savefig('../plots/ivar_map.eps', bbox_inches='tight', dpi=300, format='eps')
# plt.show()


In [ ]:
target = "bridge"

p_range_a401 = [(44.736, 44.746),
               (13.57, 13.59),
               (1.0, 1.6),
               (3.2, 6.5),
               (0.6, 1),
               (80, 140),
               (.006, .009),
               (7.5, 9.5),
               (-1e5, .9e6),
               (-4200, 1600)]

p_range_a399 = [(44.45, 44.476),
               (13.025, 13.052),
               (.75, 1.3),
               (2.5, 5.7),
               (.6, 1.3),
               (40, 180),
               (.005, .008),
               (6.4, 8),
               (-100e3, 900e3),
               (-1700, 3200 )]

    
#     labels_fil =  [r"$RA_{\rm fil}",
#               r"$DEC_{\rm fil}",
#               r"$L_{\rm fil}$",
#               r"$W_{\rm fil}$",
#               r'$\tau_{\rm fil}$',
#               r'$T_{\rm e, fil}$',
#               r'$A_{\rm D, fil}$',
              
#               r"$v_{r, avg}$"]

p_range_fil = [(44.55, 44.8),
               (13.2, 13.5),
               (8.5, 23),
               (8.5, 17),
               (.0006, 0.0018),
               (5., 7.9),
               (-3.5e4, 0.4e6),
               (-5000, 6000)]

config = "/home/gill/research/ACT/bridge/results/ajay/run_ajay_new/test13/config.yaml"
chain = "/home/gill/research/ACT/bridge/results/ajay/run_ajay_new/test13/chain.h5" 

chain = "/home/gill/research/ACT/bridge/results/test8/chain.h5"
#chain = "/home/gill/research/ACT/bridge/results/ajay/run_ajay_new/case2/chain.h5"

case = "case43"

config = f"/home/gill/research/ACT/multi-freq-bridge/configs/{case}_ajay.yaml"
chain = f"/home/gill/research/ACT/bridge/results/ajay/final_runs/{case}/chain.h5"

# change plt parameters to tex font
plt.rcParams['text.usetex'] = 1

model_tot= plotter_sim(filename=chain, 
                       target=f"{target}",
                       plot_maps=1,
                       burnin=2000,
                       thin=1,
                       fit_dust=1,
                       plot_contours=1,
                       p_range=p_range_fil,
                       plot_samples=0,
                       cf_path=config,
                       plot_converge=0)

In [ ]:
# Compton y calculation
from astropy.constants import k_B, m_p, c, sigma_T, m_e
from astropy import units as u

c1_tau = 7.211E-03
c1_tau_err = 4.81E-04
c1_Te = 8.458 * u.keV
c1_Te_K = (c1_Te / k_B).to(u.K)
c1_Te_err = 0.249 * u.keV
c1_Te_err_K = (c1_Te_err / k_B).to(u.K)
y = c1_tau * k_B * c1_Te_K / (m_e * c**2).to(u.J)
yerr = y * np.sqrt((c1_tau_err / c1_tau)**2 + (c1_Te_err_K / c1_Te_K)**2)
print(f"y = {y:.3e} +/- {yerr:.3e}")

c2_tau = 6.505E-03
c2_tau_err = 4.670E-04
c2_Te = 7.228 * u.keV
c2_Te_K = (c2_Te / k_B).to(u.K)
c2_Te_err = 0.187 * u.keV
c2_Te_err_K = (c2_Te_err / k_B).to(u.K)
y2 = c2_tau * k_B * c2_Te_K / (m_e * c**2).to(u.J)
y2err = y2 * np.sqrt((c2_tau_err / c2_tau)**2 + (c2_Te_err_K / c2_Te_K)**2)
print(f"y2 = {y2:.3e} +/- {y2err:.3e}")

fil_tau = 1.145E-03
fil_tau_err = 1.58E-04
fil_Te = 6.509 * u.keV
fil_Te_K = (fil_Te / k_B).to(u.K)
fil_Te_err = 0.350 * u.keV
fil_Te_err_K = (fil_Te_err / k_B).to(u.K)
y_fil = fil_tau * k_B * fil_Te_K / (m_e * c**2).to(u.J)
y_filerr = y_fil * np.sqrt((fil_tau_err / fil_tau)**2 + (fil_Te_err_K / fil_Te_K)**2)
print(f"y_fil = {y_fil:.3e} +/- {y_filerr:.3e}")

In [ ]:
# import numpy as np

# # Define the calculation function
# def calculate_sigma_difference(val, err, val_mfreq, err_mfreq):
#     return (val - val_mfreq) / np.sqrt(err**2 + err_mfreq**2)

# # A401 multi-frequency values
# RA_A401_mfreq_thesis = 44.741103
# RA_A401_mfreq_thesis_err = 0.001664
# DEC_A401_mfreq_thesis = 13.580485
# DEC_A401_mfreq_thesis_err = 0.002196
# beta_A401_mfreq_thesis = 1.241324
# beta_A401_mfreq_thesis_err = 0.093682
# rc_A401_mfreq_thesis = 4.362164
# rc_A401_mfreq_thesis_err = 0.493953
# e_A401_mfreq_thesis = 0.7997754231
# e_A401_mfreq_thesis_err = 0.091367
# theta_A401_mfreq_thesis = 109.403005
# theta_A401_mfreq_thesis_err = 7.142419
# y_A401_mfreq_thesis = 1.226e-04
# y_A401_mfreq_thesis_err = 8.499e-06
# y_compton_A401 = 1.260e-04  # Compton y for A401
# y_compton_A401_err = 6.00e-06  # Error for Compton y in A401

# # A399 multi-frequency values
# RA_A399_mfreq_thesis = 44.465185
# RA_A399_mfreq_thesis_err = 0.003391
# DEC_A399_mfreq_thesis = 13.039259
# DEC_A399_mfreq_thesis_err = 0.003643
# beta_A399_mfreq_thesis = 1.177231
# beta_A399_mfreq_thesis_err = 0.185405
# rc_A399_mfreq_thesis = 4.632751
# rc_A399_mfreq_thesis_err = 0.807585
# e_A399_mfreq_thesis = 1.019972
# e_A399_mfreq_thesis_err = 0.120056
# theta_A399_mfreq_thesis = 116.298848
# theta_A399_mfreq_thesis_err = 56.800183
# y_A399_mfreq_thesis = 8.249e-05
# y_A399_mfreq_thesis_err = 7.828e-06
# y_compton_A399 = 8.100e-05  # Compton y for A399
# y_compton_A399_err = 6.00e-06  # Error for Compton y in A399

# # Filament multi-frequency values
# RA_Fil_mfreq_thesis = 44.659867
# RA_Fil_mfreq_thesis_err = 0.021525
# DEC_Fil_mfreq_thesis = 13.345221
# DEC_Fil_mfreq_thesis_err = 0.034097
# L_Fil_mfreq_thesis = 15.965661
# L_Fil_mfreq_thesis_err = 2.49665
# W_Fil_mfreq_thesis = 13.135884
# W_Fil_mfreq_thesis_err = 0.982828
# y_Fil_mfreq_thesis = 1.843e-05
# y_Fil_mfreq_thesis_err = 2.267e-06
# y_compton_Fil = 1.100e-05  # Compton y for Filament
# y_compton_Fil_err = 1.750e-06  # Error for Compton y in Filament

# # Compute sigma differences
# print("=== A401 ===")
# print(f"RA_A401: {calculate_sigma_difference(44.751, 0.002, RA_A401_mfreq_thesis, RA_A401_mfreq_thesis_err):.2f} sigma")
# print(f"DEC_A401: {calculate_sigma_difference(13.572, 0.002, DEC_A401_mfreq_thesis, DEC_A401_mfreq_thesis_err):.2f} sigma")
# print(f"beta_A401: {calculate_sigma_difference(0.82, 0.05, beta_A401_mfreq_thesis, beta_A401_mfreq_thesis_err):.2f} sigma")
# print(f"rc_A401: {calculate_sigma_difference(2.6, 0.35, rc_A401_mfreq_thesis, rc_A401_mfreq_thesis_err):.2f} sigma")
# print(f"e_A401: {calculate_sigma_difference(0.82, 0.055, e_A401_mfreq_thesis, e_A401_mfreq_thesis_err):.2f} sigma")
# print(f"theta_A401: {calculate_sigma_difference(123, 8.5, theta_A401_mfreq_thesis, theta_A401_mfreq_thesis_err):.2f} sigma")
# print(f"y_A401: {calculate_sigma_difference(1.260e-04, 6.00e-06, y_A401_mfreq_thesis, y_A401_mfreq_thesis_err):.2f} sigma")
# print(f"y_compton_A401: {calculate_sigma_difference(y_compton_A401, y_compton_A401_err, y_A401_mfreq_thesis, y_A401_mfreq_thesis_err):.2f} sigma")

# print("\n=== A399 ===")
# print(f"RA_A399: {calculate_sigma_difference(44.473, 0.004, RA_A399_mfreq_thesis, RA_A399_mfreq_thesis_err):.2f} sigma")
# print(f"DEC_A399: {calculate_sigma_difference(13.03, 0.003, DEC_A399_mfreq_thesis, DEC_A399_mfreq_thesis_err):.2f} sigma")
# print(f"beta_A399: {calculate_sigma_difference(0.81, 0.095, beta_A399_mfreq_thesis, beta_A399_mfreq_thesis_err):.2f} sigma")
# print(f"rc_A399: {calculate_sigma_difference(3, 0.65, rc_A399_mfreq_thesis, rc_A399_mfreq_thesis_err):.2f} sigma")
# print(f"e_A399: {calculate_sigma_difference(0.93, 0.06, e_A399_mfreq_thesis, e_A399_mfreq_thesis_err):.2f} sigma")
# print(f"theta_A399: {calculate_sigma_difference(133, 26.5, theta_A399_mfreq_thesis, theta_A399_mfreq_thesis_err):.2f} sigma")
# print(f"y_A399: {calculate_sigma_difference(8.100e-05, 6.00e-06, y_A399_mfreq_thesis, y_A399_mfreq_thesis_err):.2f} sigma")
# print(f"y_compton_A399: {calculate_sigma_difference(y_compton_A399, y_compton_A399_err, y_A399_mfreq_thesis, y_A399_mfreq_thesis_err):.2f} sigma")

# print("\n=== Filament ===")
# print(f"RA_Fil: {calculate_sigma_difference(44.68, 0.02, RA_Fil_mfreq_thesis, RA_Fil_mfreq_thesis_err):.2f} sigma")
# print(f"DEC_Fil: {calculate_sigma_difference(13.37, 0.025, DEC_Fil_mfreq_thesis, DEC_Fil_mfreq_thesis_err):.2f} sigma")
# print(f"L_Fil: {calculate_sigma_difference(12.3, 1.55, L_Fil_mfreq_thesis, L_Fil_mfreq_thesis_err):.2f} sigma")
# print(f"W_Fil: {calculate_sigma_difference(10.8, 1.05, W_Fil_mfreq_thesis, W_Fil_mfreq_thesis_err):.2f} sigma")
# print(f"y_Fil: {calculate_sigma_difference(1.100e-05, 1.750e-06, y_Fil_mfreq_thesis, y_Fil_mfreq_thesis_err):.2f} sigma")
# print(f"y_compton_Fil: {calculate_sigma_difference(y_compton_Fil, y_compton_Fil_err, y_Fil_mfreq_thesis, y_Fil_mfreq_thesis_err):.2f} sigma")


In [ ]:
# import pandas as pd
# import numpy as np

# # For each object we want to include a set of parameters.
# # For A401 and A399 the “compton” values (taken from cell index 12) are available for all parameters.
# # For the filament compton values exist only for y.
# # We include:
# #   – RA, DEC, beta, rc, e, theta, y, T, tau, Ad for clusters (A401, A399)
# #   – RA, DEC, L, W, y, T, tau, Ad for Filament
# #
# # For each parameter we compute:
# #   diff_mfreq_vs_compton = (mfreq value) - (compton value)   [if compton exists, else NaN]
# #   diff_mfreq_thesis_vs_compton = (mfreq_thesis value) - (compton value) [if available]
# #   diff_mfreq_vs_mfreq_thesis = (mfreq value) - (mfreq_thesis value) [if available]
# #
# # Create a list of dictionaries (one per parameter per object)

# rows = []

# # --- For A401 ---
# params_A401 = {
#     "RA": {"mfreq": RA_A401_mfreq, "mfreq_err": RA_A401_mfreq_err,
#            "compton": RA_A401, "compton_err": RA_A401_err,
#            "mfreq_thesis": RA_A401_mfreq_thesis, "mfreq_thesis_err": RA_A401_mfreq_thesis_err},
#     "DEC": {"mfreq": DEC_A401_mfreq, "mfreq_err": DEC_A401_mfreq_err,
#            "compton": DEC_A401, "compton_err": DEC_A401_err,
#            "mfreq_thesis": DEC_A401_mfreq_thesis, "mfreq_thesis_err": DEC_A401_mfreq_thesis_err},
#     "beta": {"mfreq": beta_A401_mfreq, "mfreq_err": beta_A401_mfreq_err,
#            "compton": beta_A401, "compton_err": beta_A401_err,
#            "mfreq_thesis": beta_A401_mfreq_thesis, "mfreq_thesis_err": beta_A401_mfreq_thesis_err},
#     "rc": {"mfreq": rc_A401_mfreq, "mfreq_err": rc_A401_mfreq_err,
#            "compton": rc_A401, "compton_err": rc_A401_err,
#            "mfreq_thesis": rc_A401_mfreq_thesis, "mfreq_thesis_err": rc_A401_mfreq_thesis_err},
#     "e": {"mfreq": e_A401_mfreq, "mfreq_err": e_A401_mfreq_err,
#            "compton": e_A401, "compton_err": e_A401_err,
#            "mfreq_thesis": e_A401_mfreq_thesis, "mfreq_thesis_err": e_A401_mfreq_thesis_err},
#     "theta": {"mfreq": theta_A401_mfreq, "mfreq_err": theta_A401_mfreq_err,
#            "compton": theta_A401, "compton_err": theta_A401_err,
#            "mfreq_thesis": theta_A401_mfreq_thesis, "mfreq_thesis_err": theta_A401_mfreq_thesis_err},
#     "y": {"mfreq": y_A401_mfreq, "mfreq_err": y_A401_mfreq_err,
#            "compton": y_A401, "compton_err": 6e-06,   # using provided error in cell 12
#            "mfreq_thesis": y_A401_mfreq_thesis, "mfreq_thesis_err": y_A401_mfreq_thesis_err},
#     "T": {"mfreq": T_A401_mfreq, "mfreq_err": T_A401_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan},
#     "tau": {"mfreq": tau_A401_mfreq, "mfreq_err": tau_A401_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan},
#     "Ad": {"mfreq": Ad_A401_mfreq, "mfreq_err": Ad_A401_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan}
# }

# for par, vals in params_A401.items():
#     rows.append({
#         "Object": "A401",
#         "Parameter": par,
#         "mfreq": vals["mfreq"],
#         "mfreq_err": vals["mfreq_err"],
#         "compton": vals["compton"],
#         "compton_err": vals["compton_err"],
#         "mfreq_thesis": vals["mfreq_thesis"],
#         "mfreq_thesis_err": vals["mfreq_thesis_err"],
#         "diff_mfreq_vs_compton": vals["mfreq"] - vals["compton"] if not np.isnan(vals["compton"]) else np.nan,
#         "diff_mfreq_thesis_vs_compton": vals["mfreq_thesis"] - vals["compton"] if not np.isnan(vals["compton"]) else np.nan,
#         "diff_mfreq_vs_mfreq_thesis": vals["mfreq"] - vals["mfreq_thesis"] if not np.isnan(vals["mfreq_thesis"]) else np.nan
#     })

# # --- For A399 ---
# params_A399 = {
#     "RA": {"mfreq": RA_A399_mfreq, "mfreq_err": RA_A399_mfreq_err,
#            "compton": RA_A399, "compton_err": RA_A399_err,
#            "mfreq_thesis": RA_A399_mfreq_thesis, "mfreq_thesis_err": RA_A399_mfreq_thesis_err},
#     "DEC": {"mfreq": DEC_A399_mfreq, "mfreq_err": DEC_A399_mfreq_err,
#            "compton": DEC_A399, "compton_err": DEC_A399_err,
#            "mfreq_thesis": DEC_A399_mfreq_thesis, "mfreq_thesis_err": DEC_A399_mfreq_thesis_err},
#     "beta": {"mfreq": beta_A399_mfreq, "mfreq_err": beta_A399_mfreq_err,
#            "compton": beta_A399, "compton_err": beta_A399_err,
#            "mfreq_thesis": beta_A399_mfreq_thesis, "mfreq_thesis_err": beta_A399_mfreq_thesis_err},
#     "rc": {"mfreq": rc_A399_mfreq, "mfreq_err": rc_A399_mfreq_err,
#            "compton": rc_A399, "compton_err": rc_A399_err,
#            "mfreq_thesis": rc_A399_mfreq_thesis, "mfreq_thesis_err": rc_A399_mfreq_thesis_err},
#     "e": {"mfreq": e_A399_mfreq, "mfreq_err": e_A399_mfreq_err,
#            "compton": e_A399, "compton_err": e_A399_err,
#            "mfreq_thesis": e_A399_mfreq_thesis, "mfreq_thesis_err": e_A399_mfreq_thesis_err},
#     "theta": {"mfreq": theta_A399_mfreq, "mfreq_err": theta_A399_mfreq_err,
#            "compton": theta_A399, "compton_err": theta_A399_err,
#            "mfreq_thesis": theta_A399_mfreq_thesis, "mfreq_thesis_err": theta_A399_mfreq_thesis_err},
#     "y": {"mfreq": y_A399_mfreq, "mfreq_err": y_A399_mfreq_err,
#            "compton": y_A399, "compton_err": 6e-06,
#            "mfreq_thesis": y_A399_mfreq_thesis, "mfreq_thesis_err": y_A399_mfreq_thesis_err},
#     "T": {"mfreq": T_A399_mfreq, "mfreq_err": T_A399_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan},
#     "tau": {"mfreq": tau_A399_mfreq, "mfreq_err": tau_A399_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan},
#     "Ad": {"mfreq": Ad_A399_mfreq, "mfreq_err": Ad_A399_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan}
# }

# for par, vals in params_A399.items():
#     rows.append({
#         "Object": "A399",
#         "Parameter": par,
#         "mfreq": vals["mfreq"],
#         "mfreq_err": vals["mfreq_err"],
#         "compton": vals["compton"],
#         "compton_err": vals["compton_err"],
#         "mfreq_thesis": vals["mfreq_thesis"],
#         "mfreq_thesis_err": vals["mfreq_thesis_err"],
#         "diff_mfreq_vs_compton": vals["mfreq"] - vals["compton"] if not np.isnan(vals["compton"]) else np.nan,
#         "diff_mfreq_thesis_vs_compton": vals["mfreq_thesis"] - vals["compton"] if not np.isnan(vals["compton"]) else np.nan,
#         "diff_mfreq_vs_mfreq_thesis": vals["mfreq"] - vals["mfreq_thesis"] if not np.isnan(vals["mfreq_thesis"]) else np.nan
#     })

# # --- For Filament ---
# # Here compton values exist only for y; for RA, DEC, L and W we set compton to NaN.
# params_Fil = {
#     "RA": {"mfreq": RA_Fil_mfreq, "mfreq_err": RA_Fil_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": RA_Fil_mfreq_thesis, "mfreq_thesis_err": RA_Fil_mfreq_thesis_err},
#     "DEC": {"mfreq": DEC_Fil_mfreq, "mfreq_err": DEC_Fil_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": DEC_Fil_mfreq_thesis, "mfreq_thesis_err": DEC_Fil_mfreq_thesis_err},
#     "L": {"mfreq": L_Fil_mfreq, "mfreq_err": L_Fil_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": L_Fil_mfreq_thesis, "mfreq_thesis_err": L_Fil_mfreq_thesis_err},
#     "W": {"mfreq": W_Fil_mfreq, "mfreq_err": W_Fil_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": W_Fil_mfreq_thesis, "mfreq_thesis_err": W_Fil_mfreq_thesis_err},
#     "y": {"mfreq": y_Fil_mfreq, "mfreq_err": y_Fil_mfreq_err,
#            "compton": y_compton_Fil, "compton_err": 1.75e-06,
#            "mfreq_thesis": y_Fil_mfreq_thesis, "mfreq_thesis_err": y_Fil_mfreq_thesis_err},
#     "T": {"mfreq": T_Fil_mfreq, "mfreq_err": T_Fil_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan},
#     "tau": {"mfreq": tau_Fil_mfreq, "mfreq_err": tau_Fil_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan},
#     "Ad": {"mfreq": Ad_Fil_mfreq, "mfreq_err": Ad_Fil_mfreq_err,
#            "compton": np.nan, "compton_err": np.nan,
#            "mfreq_thesis": np.nan, "mfreq_thesis_err": np.nan}
# }

# for par, vals in params_Fil.items():
#     rows.append({
#         "Object": "Filament",
#         "Parameter": par,
#         "mfreq": vals["mfreq"],
#         "mfreq_err": vals["mfreq_err"],
#         "compton": vals["compton"],
#         "compton_err": vals["compton_err"],
#         "mfreq_thesis": vals["mfreq_thesis"],
#         "mfreq_thesis_err": vals["mfreq_thesis_err"],
#         "diff_mfreq_vs_compton": vals["mfreq"] - vals["compton"] if not np.isnan(vals["compton"]) else np.nan,
#         "diff_mfreq_thesis_vs_compton": vals["mfreq_thesis"] - vals["compton"] if not np.isnan(vals["compton"]) else np.nan,
#         "diff_mfreq_vs_mfreq_thesis": vals["mfreq"] - vals["mfreq_thesis"] if not np.isnan(vals["mfreq_thesis"]) else np.nan
#     })

# # Convert list of rows to DataFrame and save that to CSV.
# df_all = pd.DataFrame(rows)
# df_all.to_csv("results_comparison_all.csv", index=False)
# print("results_comparison_all.csv created successfully.")

In [ ]:
# from mpl_toolkits.axes_grid1 import make_axes_locatable


# from astropy.visualization import (ZScaleInterval,
#                                     PercentileInterval,
#                                     MinMaxInterval,
#                                     ImageNormalize,
#                                     simple_norm)
# from matplotlib.ticker import MaxNLocator

# region_center_ra = 44.5916744
# region_center_dec = 13.2999979

# box_region = ut.get_region(region_center_ra, region_center_dec, 3.1 * 2.1)

# data_map_pl = ut.imap_dim_check(enmap.read_map("/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits",
#                                                box=box_region))

# flux_factor = ut.flux_factor("pa5", 98) # (Jy/sr/uK)

# data_map_pl *= flux_factor / 1000

# fig = plt.figure(figsize=(8, 8))
# ax = fig.add_subplot(111, projection=data_map_pl.wcs)

# norm = ImageNormalize(data_map_pl, interval=ZScaleInterval())

# vmin = data_map_pl.min()
# vmax = data_map_pl.max()

# im = ax.imshow(data_map_pl, origin='lower', cmap='planck', interpolation='none',
#                 vmin=vmin, vmax=vmax)
# # Access coordinate axes
# ra = ax.coords[0]
# dec = ax.coords[1]
# color = 'black'

# x_center = region_center_ra
# y_center = region_center_dec

# # Set labels
# ra.set_axislabel('Right Ascension')
# dec.set_axislabel('Declination')

# # Ensure tick labels are in float degrees
# ra.set_major_formatter('d')
# dec.set_major_formatter('d')

# ax.invert_xaxis()

# divider = make_axes_locatable(ax)

# cbar = plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.04, label='kJy/sr')
# # cbar.ax.xaxis.set_major_locator(MaxNLocator(nbins=10))

# # make boxes of 1.9 deg x 1.9 deg around the central region and around it
# import matplotlib.patches as patches

# box_size = 2.1  # size of each box in degrees
# # Create a 3x3 grid of boxes centered on (x_center, y_center) and label them
# region_counter = 0

# for dy_offset in [box_size, 0, -box_size]:
#     for dx_offset in [box_size, 0, -box_size]:
#         lower_left = (x_center + dx_offset - box_size/2, y_center + dy_offset - box_size/2)
#         rect = patches.Rectangle(lower_left, box_size, box_size, 
#                                  edgecolor=color, facecolor='none', lw=1,
#                                  transform=ax.get_transform('world'))
#         ax.add_patch(rect)
        
#         label_x = lower_left[0] + box_size/2
#         label_y = lower_left[1] + 0.8 * box_size
        
#         if dx_offset == 0 and dy_offset == 0:
#             label1 = "Cluster pair"
#             label2 = "Abell 401"
#             label3 = "Abell 399"
#             # ax.text(label_x, 0.89*label_y, label1, color=color, fontsize=14,
#             #     ha='center', va='bottom', transform=ax.get_transform('world'))
#             ax.text(label_x, 0.99*label_y, label2, color=color, fontsize=14,
#                 ha='center', va='bottom', transform=ax.get_transform('world'))
#             ax.text(0.995*label_x, 0.91*label_y, label3, color=color, fontsize=14,
#                 ha='center', va='bottom', transform=ax.get_transform('world'))

#         else:
#             label = f"Region {region_counter}"
#             region_counter += 1
        
#             ax.text(label_x, label_y, label, color=color, fontsize=14,
#                     ha='center', va='bottom', transform=ax.get_transform('world'))

# plt.savefig("../plots/region_map.pdf", bbox_inches='tight', dpi=300, format='pdf')
# plt.show()

In [ ]:
# def map_maker(mcmc_fname, cf_name, labels, labels_no_tex, burnin=1, thin=1, 
#               plot_converge=False, print_cf=False, plot_samples=False):
    
#     # Read chain samples and print diagnostics
#     sampler = emcee.backends.HDFBackend(mcmc_fname)
#     samples = sampler.get_chain(discard=burnin, flat=True, thin=thin)  
#     samples_unflat = sampler.get_chain(discard=burnin)
#     acc_frac = sampler.accepted / sampler.iteration
    
#     print("*Average acceptance fraction is: {:.2f}%".format(np.mean(acc_frac)*100))
    
#     ndim = len(labels)
#     plt.rc('text', usetex=True)
#     plt.rc('font', family='sans-serif', size=20)
#     print("\nNumber of iterations: {:.0f}".format(samples.shape[0] / sampler.shape[0]))
#     if plot_converge:
#         print("Convergence plot")
#         converge_plot(sampler, labels)
    
#     cf = ut.get_config_file(cf_name)
#     if print_cf:
#         for key, value in cf.items():
#             print(key, ":", value)
    
#     # Get region and common WCS from a reference ACT map
#     region = ut.get_region(cf['region_center_ra'], cf['region_center_dec'], cf['region_width'])
#     dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"
#     dire_data_planck = "/home/gill/research/ACT/bridge/data_paper/data/data/planck_no_reproj"
   
#     # Use one ACT map as the reference for Wcs and data_shape
#     data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
#                                                box=region))
    
#     #data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits")) 

#     data_wcs = data_ref.wcs
#     data_shape = data_ref.shape

#     # Compute theta from samples (for all parameters, here for each label)
#     theta = []
#     for idx, label in enumerate(labels):
#         mcmc_run = np.percentile(samples[:, idx], [16, 50, 84])
#         err = 0.5 * (mcmc_run[2] - mcmc_run[0])
#         data = samples[:, idx]
#         n = len(data)
#         iqr = np.percentile(data, 75) - np.percentile(data, 25)
#         bin_width = 2 * iqr * n**(-1/3)
#         num_bins = int((np.max(data) - np.min(data)) / bin_width)
#         hist, bin_edges = np.histogram(data, bins=num_bins, density=True)
#         mode_bin = np.argmax(hist)
#         mode_value = 0.5*(bin_edges[mode_bin] + bin_edges[mode_bin+1])
#         theta.append(mode_value)
#         print("{}: {:.6f}, +/- {:.6f}".format(label, mode_value, err))
    
#     # Prepare cluster and filament models
#     #RA_A401 
#     #DEC_A401
    

#     c1 = model.Cluster(theta=theta, name="abell401", model_choice="fit_vavg")
#     c2 = model.Cluster(theta=theta, name="abell399", model_choice="fit_vavg")
#     fil = model.Filament(theta=theta, model_choice="fit_vavg")
    
#     ra_c1_fit = 131.901050
#     dec_c1_fit = 147.427822 

#     # Set up frequency-dependent dictionary: keys as string frequencies
#     freqs_params = {
#         '30': {'freq':30, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_030_coadd_map_srcfree.fits"},
#         '44': {'freq':44, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_044_coadd_map_srcfree.fits"},
#         '70': {'freq':70, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_070_coadd_map_srcfree.fits"},
#         '100':{'freq':100, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_100_coadd_map_srcfree.fits"},
#         '143':{'freq':143, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_143_coadd_map_srcfree.fits"},
#         '217':{'freq':217, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_217_coadd_map_srcfree.fits"},
#         '353':{'freq':353, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_353_coadd_map_srcfree.fits"},
#         '545':{'freq':545, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
#                'file':"planck_npipe_545_coadd_map_srcfree.fits"},
#         '98': {'freq':98,  'array':'pa6',  'inst':'act',     'dir':dire_data, 
#                'file':"act_cut_dr6v2_pa6_f098_4way_coadd_map_srcfree.fits"},
#         '150':{'freq':150, 'array':'pa6',  'inst':'act',     'dir':dire_data, 
#                'file':"act_cut_dr6v2_pa6_f150_4way_coadd_map_srcfree.fits"},
#         '220':{'freq':220, 'array':'pa4',  'inst':'act',     'dir':dire_data, 
#                'file':"act_cut_dr6v2_pa4_f220_4way_coadd_map_srcfree.fits"}
#     }
    
#     # Containers for results that will be used for plotting later
#     ref_data_dict = {}
#     model_tot_dict = {}
    
#     # Create common grid (assumed same for all frequencies)
#     xgrid, ygrid = np.meshgrid(np.arange(data_shape[1]), np.arange(data_shape[0]))
    
#     # Loop over frequencies
#     for key, params in freqs_params.items():
#         freq = params['freq']
#         array = params['array']
#         inst = params['inst']
#         map_file = f"{params['dir']}/{params['file']}"
        
#         ref_map = enmap.read_map(map_file, box=region)
#         #ref_map = enmap.read_map(map_file)
#         ref_data_curr = ut.imap_dim_check(ref_map)
#         # Multiply by flux factor
#         flux_factor = ut.flux_factor(array, freq)
#         ref_data_curr *= flux_factor
        
#         # Build SZ models using the same grid
#         c1_model = c1.szmodel(frequency=freq, array=array, z=cf['c1_z'], muo=cf['c1_muo'],
#                                xgrid=xgrid, ygrid=ygrid, ellipticity_type="numerator")
#         c2_model = c2.szmodel(frequency=freq, array=array, z=cf['c2_z'], muo=cf['c2_muo'],
#                                xgrid=xgrid, ygrid=ygrid, ellipticity_type="numerator")
#         fil_model = fil.szmodel(frequency=freq, array=array, z=cf['fil_z'], muo=cf['fil_muo'],
#                                  xgrid=xgrid, ygrid=ygrid)
#         total_model = c1_model + c2_model + fil_model
        
#         # Convolve with beam
#         beam = ut.get_2d_beam(data_shape=ref_data_curr.shape, freq=freq, array=array, 
#                               inst=inst, version="dr6v2", data_wcs=data_wcs)
#         model_tot = np.real(np.fft.ifft2(np.fft.fft2(total_model) * beam))
        
#         # Get SZ signals at the set (RA,Dec) pixel
#        #  c1_model_SZ = enmap.ndmap(c1_model[1], data_ref.wcs)
#        #  dust_model_SZ = enmap.ndmap(c1_model[2], data_ref.wcs)
#        #  total_model_pix = enmap.ndmap(c1_model, data_ref.wcs)
#        #  c1_signal = c1_model_SZ.at([dec_c1_fit, ra_c1_fit], unit="pix")
#        #  dust_signal = dust_model_SZ.at([dec_c1_fit, ra_c1_fit], unit="pix")
#        #  tot_signal = total_model_pix.at([dec_c1_fit, ra_c1_fit], unit="pix")
        
#         # Save into dictionaries
#         ref_data_dict[key] = ref_data_curr
#         model_tot_dict[key] = model_tot
#        #  c1_signals[key] = c1_signal
#        #  dust_signals[key] = dust_signal
#        #  total_model_signals[key] = tot_signal
        
#     # PLOTTING: Define instruments and array mapping for plot titles
#     inst_dict = {'98':"ACT", "150":"ACT", "220":"ACT", 
#                  "30":"Planck", "44":"Planck", "70":"Planck",
#                  "100":"Planck", "143":"Planck", 
#                  "217":"Planck", "353": "Planck", "545":"Planck"}
#     array_dict = {'98':'pa5', "150":'pa5', "220":'pa4',
#                   "30":'npipe', "44":'npipe', "70":'npipe',
#                   "100":'npipe', "143":'npipe', "217":'npipe',
#                   "353":'npipe', "545":'npipe'}
    
#     lab = r"$I$ [kJy / sr]"
#     # Order for plotting (only those keys present in freqs_params)
#     plot_keys = ['30','44','70','98','100','143','150','217','220','353','545']
    
#     for key in plot_keys:
#         # Skip any frequencies not processed
#         if key not in ref_data_dict:
#             continue
        
#         fig, axes = plt.subplots(1, 3, figsize=(15, 5), 
#                                  subplot_kw={'projection': data_ref.wcs})

#         cmap = "planck"
#         pad = 0.17

#         ref_data_curr = ref_data_dict[key]
#         model_tot = model_tot_dict[key]
        
#         # Calculate symmetric color scale limits
#         data_scaled = ref_data_curr/1e3
#         model_scaled = model_tot/1e3
#         resi = (ref_data_curr - model_tot)/1e3
        
#         # Use symmetric scaling for better visualization
#         data_abs_max = np.max(np.abs(data_scaled))
#         model_abs_max = np.max(np.abs(model_scaled))
#         resi_abs_max = np.max(np.abs(resi))
        
#         # Set up coordinate formatting for all subplots
#         for i, ax in enumerate(axes):
#             ra = ax.coords[0]
#             dec = ax.coords[1] 
#             ra.set_axislabel('Right Ascension')
#             dec.set_axislabel('Declination') 
#             ra.set_major_formatter('d')
#             dec.set_major_formatter('d') 
#             ax.invert_xaxis()
        
#         # Plot data
#         im1 = axes[0].imshow(data_scaled, origin='lower', cmap=cmap, 
#                            vmin=-data_abs_max, vmax=data_abs_max)
#         plt.colorbar(im1, ax=axes[0], orientation="horizontal", pad=pad, label=lab)
#         axes[0].set_title(f"Data: {inst_dict[key]} {array_dict[key]} {key} GHz", fontsize=16)
        
#         # Plot model
#         im2 = axes[1].imshow(model_scaled, origin='lower', cmap=cmap,
#                            vmin=-model_abs_max, vmax=model_abs_max)
#         axes[1].set_yticklabels([])
#         plt.colorbar(im2, ax=axes[1], orientation="horizontal", pad=pad, label=lab)
#         axes[1].set_title(f"Model: {inst_dict[key]} {array_dict[key]} {key} GHz", fontsize=16)
        
#         # Plot residual
#         resi = enmap.enmap(resi, data_ref.wcs)
#         im3 = axes[2].imshow(resi, origin='lower', cmap=cmap,
#                            vmin=-resi_abs_max, vmax=resi_abs_max)
#         axes[2].set_yticklabels([])
#         plt.colorbar(im3, ax=axes[2], orientation="horizontal", pad=pad, label=lab)
#         axes[2].set_title(f"Residual: {inst_dict[key]} {array_dict[key]} {key} GHz", fontsize=16)
        
#         plt.tight_layout()
#         plt.savefig(f"../plots/bridge_{key}.pdf", bbox_inches='tight', dpi=400)
#         plt.show()
    
# #     # Build arrays of signals (ordered by frequency key order)
# #     ordered_keys = [k for k in plot_keys if k in c1_signals]
# #     c1_sz = np.array([c1_signals[k] for k in ordered_keys])
# #     dust_sz = np.array([dust_signals[k] for k in ordered_keys])
# #     tot_sz = np.array([total_model_signals[k] for k in ordered_keys])
    
#     # Return as in the original function (plus additional outputs if needed)
# #     return c1_sz, dust_sz, tot_sz, c1_model[3], c1_model[4]


In [ ]:
from pixell import colorize
colorize.mpl_register("planck")

In [ ]:
def map_maker_improved(mcmc_fname, cf_name, labels, labels_no_tex, burnin=1, thin=1, 
                       plot_converge=False, print_cf=False, plot_samples=False):
    
    from astropy import units as u
    from astropy.visualization.wcsaxes import SphericalCircle
    
    # Read chain samples and print diagnostics
    sampler = emcee.backends.HDFBackend(mcmc_fname)
    samples = sampler.get_chain(discard=burnin, flat=True, thin=thin)  
    samples_unflat = sampler.get_chain(discard=burnin)
    acc_frac = sampler.accepted / sampler.iteration
    
    print("*Average acceptance fraction is: {:.2f}%".format(np.mean(acc_frac)*100))
    
    ndim = len(labels)
    plt.rc('text', usetex=True)
    plt.rc('font', family='sans-serif', size=20)
    print("\nNumber of iterations: {:.0f}".format(samples.shape[0] / sampler.shape[0]))
    if plot_converge:
        print("Convergence plot")
        converge_plot(sampler, labels)
    
    cf = ut.get_config_file(cf_name)
    if print_cf:
        for key, value in cf.items():
            print(key, ":", value)    
    # Get region and common WCS from a reference ACT map
    region = ut.get_region(cf['region_center_ra'], cf['region_center_dec'], cf['region_width'])
    dire_data = "/home/gill/research/ACT/bridge/data_paper/data/data/act_no_reproj"
    dire_data_planck = "/home/gill/research/ACT/bridge/data_paper/data/data/planck_no_reproj"
   
    # Use one ACT map as the reference for Wcs and data_shape
    data_ref = ut.imap_dim_check(enmap.read_map(f"{dire_data}/act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits", 
                                               box=region))
    
    data_wcs = data_ref.wcs
    data_shape = data_ref.shape

    # Compute theta from samples (for all parameters, here for each label)
    theta = []
    for idx, label in enumerate(labels):
        mcmc_run = np.percentile(samples[:, idx], [16, 50, 84])
        err = 0.5 * (mcmc_run[2] - mcmc_run[0])
        data = samples[:, idx]
        n = len(data)
        iqr = np.percentile(data, 75) - np.percentile(data, 25)
        bin_width = 2 * iqr * n**(-1/3)
        num_bins = int((np.max(data) - np.min(data)) / bin_width)
        hist, bin_edges = np.histogram(data, bins=num_bins, density=True)
        mode_bin = np.argmax(hist)
        mode_value = 0.5*(bin_edges[mode_bin] + bin_edges[mode_bin+1])
        theta.append(mcmc_run[1])  # Use median as theta
        print("{}: {:.6f}, +/- {:.6f}".format(label, mcmc_run[1], err))
    
    # Prepare cluster and filament models
    c1 = model.Cluster(theta=theta, name="abell401", model_choice="fit_vavg")
    c2 = model.Cluster(theta=theta, name="abell399", model_choice="fit_vavg")
    fil = model.Filament(theta=theta, model_choice="fit_vavg")

    ra_c1_idx = 0
    dec_c1_idx = 1
    ra_c2_idx = 10
    dec_c2_idx = 11
    ra_fil_idx = 20
    dec_fil_idx = 21

    ra_c1_fit = theta[ra_c1_idx]
    dec_c1_fit = theta[dec_c1_idx]
    ra_c2_fit = theta[ra_c2_idx]
    dec_c2_fit = theta[dec_c2_idx]
    ra_fil_fit = theta[ra_fil_idx]
    dec_fil_fit = theta[dec_fil_idx]

    # Set up frequency-dependent dictionary: keys as string frequencies
    freqs_params = {
        '30': {'freq':30, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_030_coadd_map_srcfree.fits"},
        '44': {'freq':44, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_044_coadd_map_srcfree.fits"},
        '70': {'freq':70, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_070_coadd_map_srcfree.fits"},
        '100':{'freq':100, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_100_coadd_map_srcfree.fits"},
        '143':{'freq':143, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_143_coadd_map_srcfree.fits"},
        '217':{'freq':217, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_217_coadd_map_srcfree.fits"},
        '353':{'freq':353, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_353_coadd_map_srcfree.fits"},
        '545':{'freq':545, 'array':'npipe', 'inst':'planck', 'dir':dire_data_planck, 
               'file':"planck_npipe_545_coadd_map_srcfree.fits"},
        '98': {'freq':98,  'array':'pa5',  'inst':'act',     'dir':dire_data, 
               'file':"act_cut_dr6v2_pa5_f098_4way_coadd_map_srcfree.fits"},
        '150':{'freq':150, 'array':'pa5',  'inst':'act',     'dir':dire_data, 
               'file':"act_cut_dr6v2_pa5_f150_4way_coadd_map_srcfree.fits"},
        '220':{'freq':220, 'array':'pa4',  'inst':'act',     'dir':dire_data, 
               'file':"act_cut_dr6v2_pa4_f220_4way_coadd_map_srcfree.fits"}
    }
    
    # Containers for results that will be used for plotting later
    ref_data_dict = {}
    model_tot_dict = {}
    c1_sz_signals = {}
    c2_sz_signals = {}
    fil_sz_signals = {}
    c1_dust_signals = {}
    c2_dust_signals = {}
    fil_dust_signals = {}

    total_sz_signals = {}
    
    # Create common grid (assumed same for all frequencies)
    xgrid, ygrid = np.meshgrid(np.arange(data_shape[1]), np.arange(data_shape[0]))
    
    # Loop over frequencies
    for key, params in freqs_params.items():
        freq = params['freq']
        array = params['array']
        inst = params['inst']
        map_file = f"{params['dir']}/{params['file']}"
        
        ref_map = enmap.read_map(map_file, box=region)
        ref_data_curr = ut.imap_dim_check(ref_map)
        # Multiply by flux factor
        flux_factor = ut.flux_factor(array, freq)
        ref_data_curr *= flux_factor
        
        # Build SZ models using the same grid
        c1_model = c1.szmodel(frequency=freq, array=array, z=cf['c1_z'], muo=cf['c1_muo'],
                               xgrid=xgrid, ygrid=ygrid, ellipticity_type="numerator")
        c2_model = c2.szmodel(frequency=freq, array=array, z=cf['c2_z'], muo=cf['c2_muo'],
                               xgrid=xgrid, ygrid=ygrid, ellipticity_type="numerator")
        fil_model = fil.szmodel(frequency=freq, array=array, z=cf['fil_z'], muo=cf['fil_muo'],
                                 xgrid=xgrid, ygrid=ygrid)
        total_model = c1_model[0] + c2_model[0] + fil_model[0]
        
        # Convolve with beam
        beam = ut.get_2d_beam(data_shape=ref_data_curr.shape, freq=freq, array=array, 
                              inst=inst, version="dr6v2", data_wcs=data_wcs)
        model_tot = np.real(np.fft.ifft2(np.fft.fft2(total_model) * beam))

        # Get SZ signals at the set (RA,Dec) pixel
        c1_model_SZ = enmap.ndmap(c1_model[1], data_ref.wcs)
        dust_model_SZ = enmap.ndmap(c1_model[2], data_ref.wcs)
        c1_signal_sz = c1_model_SZ.at([dec_c1_fit, ra_c1_fit], unit="pix")
        c1_signal_dust = dust_model_SZ.at([dec_c1_fit, ra_c1_fit], unit="pix")

        c2_model_SZ = enmap.ndmap(c2_model[1], data_ref.wcs)
        dust_model_SZ = enmap.ndmap(c2_model[2], data_ref.wcs)
        c2_signal_sz = c2_model_SZ.at([dec_c2_fit, ra_c2_fit], unit="pix")
        c2_signal_dust = dust_model_SZ.at([dec_c2_fit, ra_c2_fit], unit="pix")

        fil_model_SZ = enmap.ndmap(fil_model[1], data_ref.wcs)
        dust_model_SZ = enmap.ndmap(fil_model[2], data_ref.wcs)
        fil_signal_sz = fil_model_SZ.at([dec_fil_fit, ra_fil_fit], unit="pix")
        fil_signal_dust = dust_model_SZ.at([dec_fil_fit, ra_fil_fit], unit="pix")
        
        # Save into dictionaries
        ref_data_dict[key] = ref_data_curr
        model_tot_dict[key] = model_tot
        c1_sz_signals[key] = c1_signal_sz
        c2_sz_signals[key] = c2_signal_sz
        fil_sz_signals[key] = fil_signal_sz
        c1_dust_signals[key] = c1_signal_dust
        c2_dust_signals[key] = c2_signal_dust
        fil_dust_signals[key] = fil_signal_dust

    # make a dictionary with vmin and vmax for each frequency
    vmin_vmax_dict = {}

    vmin_vmax_dict['30'] = {'vmin': -10, 'vmax': 10}
    vmin_vmax_dict['44'] = {'vmin': -15, 'vmax': 15}
    vmin_vmax_dict['70'] = {'vmin': -50, 'vmax': 50}
    vmin_vmax_dict['98'] = {'vmin': -60, 'vmax': 60}
    vmin_vmax_dict['100'] = {'vmin': -100, 'vmax': 100}
    vmin_vmax_dict['143'] = {'vmin': -150, 'vmax': 150}
    vmin_vmax_dict['150'] = {'vmin': -150, 'vmax': 150}
    vmin_vmax_dict['217'] = {'vmin': -300, 'vmax': 300}
    vmin_vmax_dict['220'] = {'vmin': -300, 'vmax': 300}
    vmin_vmax_dict['353'] = {'vmin': -1000, 'vmax': 1000}
    vmin_vmax_dict['545'] = {'vmin': 0, 'vmax': 3000}

    vmin_vmax_model_dict = {
        '30': {'vmin': -2, 'vmax': 2},
        '44': {'vmin': -6, 'vmax': 6},
        '70': {'vmin': -25, 'vmax': 25},
        '98': {'vmin': -80, 'vmax': 80},
        '100': {'vmin': -35, 'vmax': 35},
        '143': {'vmin': -40, 'vmax': 40},
        '150': {'vmin': -40, 'vmax': 40},
        '217': {'vmin': -30, 'vmax': 30},
        '220': {'vmin': -30, 'vmax': 30},
        '353': {'vmin': 0, 'vmax': 120},
        '545': {'vmin': 0, 'vmax': 120}
    }

    beam_fwhm_dict = {
        '30': 32.4, 
        '44': 27.1, 
        '70': 13.3,
        '98': 2.1, 
        '100': 9.7, 
        '143': 7.3,
        '150': 1.4, 
        '217': 5.0, 
        '220': 1.0,
        '353': 4.9, 
        '545': 4.60
    } # units of arcmin
        
    # PLOTTING: Define instruments and array mapping for plot titles
    inst_dict = {'98':"ACT", "150":"ACT", "220":"ACT", 
                 "30":"Planck", "44":"Planck", "70":"Planck",
                 "100":"Planck", "143":"Planck", 
                 "217":"Planck", "353": "Planck", "545":"Planck"}
    array_dict = {'98':'pa5', "150":'pa5', "220":'pa4',
                  "30":'npipe', "44":'npipe', "70":'npipe',
                  "100":'npipe', "143":'npipe', "217":'npipe',
                  "353":'npipe', "545":'npipe'}
    
    # Order for plotting (only those keys present in freqs_params)
    plot_keys = ['30','44','70','98','100','143','150','217','220','353','545']
    
    for key in plot_keys:
        # Skip any frequencies not processed
        if key not in ref_data_dict:
            continue
        
        # Get data for this frequency
        ref_data_curr = ref_data_dict[key] / 1e3  # Convert to kJy/sr
        model_tot = model_tot_dict[key] / 1e3     # Convert to kJy/sr
        residual = ref_data_curr - model_tot

       # save the model_tot with WCS information
        filename = f"/home/gill/research/ACT/paper/models/models_july12/{key}_model.fits"
        model_tot_with_wcs = enmap.ndmap(model_tot, data_ref.wcs)
        enmap.write_map(filename, model_tot_with_wcs)
        
        vmin = vmin_vmax_dict[key]['vmin']
        vmax = vmin_vmax_dict[key]['vmax']

        vmin_model = vmin_vmax_model_dict[key]['vmin']
        vmax_model = vmin_vmax_model_dict[key]['vmax']
        
        # Create figure with individual subplots for better control
        fig = plt.figure(figsize=(15, 5))
        
        # Data panel
        ax1 = fig.add_subplot(131, projection=data_ref.wcs)
        im1 = ax1.imshow(ref_data_curr, origin='lower', cmap='planck', 
                         vmin=vmin, vmax=vmax, interpolation='none')
        ax1.coords[0].set_axislabel('Right Ascension')
        ax1.coords[1].set_axislabel('Declination')
        ax1.coords[0].set_major_formatter('d')
        ax1.coords[1].set_major_formatter('d')
        ax1.invert_xaxis()
        ax1.set_title(f'Data: {inst_dict[key]} {key} GHz', fontsize=20, pad=20)
        
        # Add beam circle to data panel
        beam_radius_arcmin = beam_fwhm_dict[key] / 2
        # Calculate offset based on beam size (larger beams get larger offsets)
        # Scale the offset with beam size, with a minimum offset for small beams
        ra0, dec0 = cf['region_center_ra'] - 0.7, cf['region_center_dec'] + 0.7

        sky_center = SkyCoord(ra0, dec0, unit='deg', frame='icrs')

        beam_circle = SphericalCircle(sky_center, beam_radius_arcmin * u.arcmin,
                                transform=ax1.get_transform('icrs'),
                               edgecolor='black', facecolor='none', linewidth=2)
        ax1.add_patch(beam_circle)

        # Model panel
        ax2 = fig.add_subplot(132, projection=data_ref.wcs)
        im2 = ax2.imshow(model_tot, origin='lower', cmap='planck', 
                         vmin=vmin_model, vmax=vmax_model, interpolation='none')
        ax2.coords[0].set_axislabel('Right Ascension')
        ax2.coords[1].set_axislabel('Declination')
        ax2.coords[0].set_major_formatter('d')
        ax2.coords[1].set_major_formatter('d')
        ax2.invert_xaxis()
        ax2.set_title(f'Model: {inst_dict[key]} {key} GHz', fontsize=20, pad=20)
        
        # Add beam circle to model panel
        beam_circle = SphericalCircle(sky_center, beam_radius_arcmin * u.arcmin,
                        transform=ax2.get_transform('icrs'),
                       edgecolor='black', facecolor='none', linewidth=2)
        ax2.add_patch(beam_circle)
        
        # Residual panel
        ax3 = fig.add_subplot(133, projection=data_ref.wcs)
        im3 = ax3.imshow(residual, origin='lower', cmap='planck', 
                 vmin=vmin, vmax=vmax, interpolation='none')
        ax3.coords[0].set_axislabel('Right Ascension')
        ax3.coords[1].set_axislabel('Declination')
        ax3.coords[0].set_major_formatter('d')
        ax3.coords[1].set_major_formatter('d')
        ax3.invert_xaxis()
        ax3.set_title(f'Residual: {inst_dict[key]} {key} GHz', fontsize=20, pad=20)
        
        # Add beam circle to residual panel
        beam_circle = SphericalCircle(sky_center, beam_radius_arcmin * u.arcmin,
                        transform=ax3.get_transform('icrs'),
                       edgecolor='black', facecolor='none', linewidth=2)
        ax3.add_patch(beam_circle)
        
        # Add colorbars
        plt.colorbar(im1, ax=ax1, orientation='horizontal', pad=0.25, 
                     label=r'$I$ [kJy/sr]', fraction=0.046)
        plt.colorbar(im2, ax=ax2, orientation='horizontal', pad=0.25, 
                     label=r'$I$ [kJy/sr]', fraction=0.046)
        plt.colorbar(im3, ax=ax3, orientation='horizontal', pad=0.25, 
                     label=r'$I$ [kJy/sr]', fraction=0.046)
        
        # plt.tight_layout()
        plt.savefig(f"../plots/bridge_{key}.pdf", bbox_inches='tight', dpi=300, format='pdf')
        plt.show()
    
#     ordered_keys = [k for k in plot_keys if k in c1_signals]
#     c1_sz = np.array([c1_signals[k] for k in ordered_keys])
#     dust_sz = np.array([dust_signals[k] for k in ordered_keys])
#     tot_sz = np.array([total_model_signals[k] for k in ordered_keys])
    
#     # Return as in the original function (plus additional outputs if needed)
# #     return c1_sz, dust_sz, tot_sz, c1_model[3], c1_model[4]


In [ ]:
labels = [r'$RA_{\rm A401}$', 
            r'$DEC_{\rm A401}$', 
            r'$\beta_{\rm A401}$', 
            r'$r_{\rm c, A401}$ [$^{\prime}$]', 
            r'$e_{\rm A401}$', 
            r'$\theta_{\rm A401}$', 
            r'$\tau_{\rm A401}$', 
            r'$T_{\rm e, A401}$', 
            r'$A_{\rm D, A401}$',
            r"$v_{r, A401}$",
            
            r'$RA_{\rm A399}$', 
            r'$DEC_{\rm A399}$', 
            r'$\beta_{\rm A399}$', 
            r'$r_{\rm c, A399}$ [$^{\prime}$]', 
            r'$e_{\rm A399}$', 
            r'$\theta_{\rm A399}$', 
            r'$\tau_{\rm A399}$', 
            r'$T_{\rm e, A399}$', 
            r'$A_{\rm D, A399}$',
            r"$v_{r, A399}$",

            r"$RA_{\rm fil}$",
            r"$DEC_{\rm fil}$",
            r"$L_{\rm fil}$",
            r"$W_{\rm fil}$",
            r'$\tau_{\rm fil}$',
            r'$T_{\rm e, fil}$',
            r'$A_{\rm D, fil}$',
            
            r"$v_{r, fil}$"]

labels_no_tex = [
    "RA_A401",
    "DEC_A401",
    "beta_A401",
    "r_c_A401 [']",
    "e_A401",
    "theta_A401",
    "tau_A401",
    "T_e_A401",
    "A_D_A401",
    
    "RA_A399",
    "DEC_A399",
    "beta_A399",
    "r_c_A399 [']",
    "e_A399",
    "theta_A399",
    "tau_A399",
    "T_e_A399",
    "A_D_A399",
    
    "RA_fil",
    "DEC_fil",
    "L_fil",
    "W_fil",
    "tau_fil",
    "T_e_fil",
    "A_D_fil",
    
    "v_r_avg"
]

# chain = "/home/gill/research/ACT/bridge/results/test8/chain.h5"
# config = "/home/gill/research/ACT/bridge/results/test8/config.yaml"

config = "/home/gill/research/ACT/multi-freq-bridge/configs/case23_ajay.yaml"
chain = "/home/gill/research/ACT/bridge/results/ajay/final_runs/case23/chain.h5"

#config = "/home/gill/research/ACT/bridge/results/ajay/run_ajay_new/test11/config.yaml"
#chain = "/home/gill/research/ACT/bridge/results/ajay/run_ajay_new/test11/chain.h5"

_ = map_maker_improved(mcmc_fname=chain,
                       cf_name=config,
                       labels=labels,
                       labels_no_tex=labels_no_tex,
                       burnin=50000,
                       thin=10, 
                       plot_converge=False,
                       plot_samples=0)

# sampler = emcee.backends.HDFBackend(chain)
# converge_plot(sampler, labels_no_tex)

# theta_fit, err_fit = get_params(mcmc_fname=chain,
#                                 cf_name=config,
#                                 labels=labels,
#                                 labels_no_tex=labels_no_tex,
#                                 burnin=1,
#                                 thin=1, 
#                                 plot_converge=0,
#                                 plot_samples=0)

# for idx, theta in enumerate(theta_fit):
#     print(theta, err_fit[idx])


In [ ]:
chain = "/home/gill/research/ACT/bridge/results/ajay/final_runs/case23/chain.h5"
sampler = emcee.backends.HDFBackend(chain)
samples = sampler.get_chain(discard=50000, flat=True, thin=10000)

In [ ]:
len(samples)

In [ ]:
labels = [r'$RA_{\rm A401}$', 
            r'$DEC_{\rm A401}$', 
            r'$\beta_{\rm A401}$', 
            r'$r_{\rm c, A401}$ [$^{\prime}$]', 
            r'$e_{\rm A401}$', 
            r'$\theta_{\rm A401}$', 
            r'$\tau_{\rm A401}$', 
            r'$T_{\rm e, A401}$', 
            r'$A_{\rm D, A401}$',
            r"$v_{r, A401}$",
            
            r'$RA_{\rm A399}$', 
            r'$DEC_{\rm A399}$', 
            r'$\beta_{\rm A399}$', 
            r'$r_{\rm c, A399}$ [$^{\prime}$]', 
            r'$e_{\rm A399}$', 
            r'$\theta_{\rm A399}$', 
            r'$\tau_{\rm A399}$', 
            r'$T_{\rm e, A399}$', 
            r'$A_{\rm D, A399}$',
            r"$v_{r, A399}$",

            r"$RA_{\rm fil}$",
            r"$DEC_{\rm fil}$",
            r"$L_{\rm fil}$",
            r"$W_{\rm fil}$",
            r'$\tau_{\rm fil}$',
            r'$T_{\rm e, fil}$',
            r'$A_{\rm D, fil}$',
            
            r"$v_{r, avg}$"]

const_c = 299792458.0 # m / s
const_k_B = 1.380649e-23 # J / K
const_h = 6.626070149999999e-25 # J / GHz

c1_signals_tSZ = []
c2_signals_tSZ = []
fil_signals_tSZ = []

c1_signals_kSZ = []
c2_signals_kSZ = []
fil_signals_kSZ = []

c1_signals_total = []
c2_signals_total = []
fil_signals_total = []

nfreq_samps = 500

for sample in samples:
    c1_tau = sample[6]
    c1_Te = sample[7]
    c1_A_D = sample[8]  
    c1_v = sample[9]

    c2_tau = sample[16]
    c2_Te = sample[17]
    c2_A_D = sample[18]
    c2_v = sample[19]

    fil_tau = sample[24]
    fil_Te = sample[25]
    fil_A_D = sample[26]
    fil_v = sample[27]

    # cluster 1 (tSZ)
    SZ_params_c1_tSZ = SZ.parameters()
    SZ_params_c1_tSZ.betao = 0.001233586736861806
    SZ_params_c1_tSZ.runmode = 'full'
    SZ_params_c1_tSZ.T_order = 10
    SZ_params_c1_tSZ.beta_order = 2

    c1_muo = -0.5566122489553409
    c1_z = 0.073664
    c1_vc_idx = 0
    c1_Te_idx = c1_Te
    c1_tau_idx = c1_tau

    SZ_params_c1_tSZ.muo = c1_muo
    SZ_params_c1_tSZ.Dtau = c1_tau_idx
    SZ_params_c1_tSZ.Te = c1_Te_idx
    SZ_params_c1_tSZ.betac = np.abs(c1_vc_idx) * 1000 / const_c
    SZ_params_c1_tSZ.set_x_array(0.1, 20, nfreq_samps)  

    if c1_vc_idx < 0:
        SZ_params_c1_tSZ.muc = -1
    else:
        SZ_params_c1_tSZ.muc = 1

    # Set higher order terms to zero
    SZ_params_c1_tSZ.means_assign_omegas(0, 0, 0)
    SZ_params_c1_tSZ.means_assign_sigmas(0, 0, 0)
    SZ_params_c1_tSZ.means_kappa = 0 

    SZ_c1_signal_tSZ = SZ.compute_combo(SZ_params_c1_tSZ, DI=True)  # Jy/sr
    c1_signals_tSZ.append(SZ_c1_signal_tSZ)

    # cluster 1 (kSZ)
    SZ_params_c1_kSZ = SZ.parameters()
    SZ_params_c1_kSZ.betao = 0.001233586736861806
    SZ_params_c1_kSZ.runmode = 'full'
    SZ_params_c1_kSZ.T_order = 10
    SZ_params_c1_kSZ.beta_order = 2

    c1_vc_idx = c1_v
    c1_Te_idx = 1e-10
    c1_tau_idx = c1_tau

    SZ_params_c1_kSZ.muo = c1_muo
    SZ_params_c1_kSZ.Dtau = c1_tau_idx
    SZ_params_c1_kSZ.Te = c1_Te_idx
    SZ_params_c1_kSZ.betac = np.abs(c1_vc_idx) * 1000 / const_c
    SZ_params_c1_kSZ.set_x_array(0.1, 20, nfreq_samps)

    if c1_vc_idx < 0:
        SZ_params_c1_kSZ.muc = -1
    else:
        SZ_params_c1_kSZ.muc = 1

    # Set higher order terms to zero
    SZ_params_c1_kSZ.means_assign_omegas(0, 0, 0)
    SZ_params_c1_kSZ.means_assign_sigmas(0, 0, 0)
    SZ_params_c1_kSZ.means_kappa = 0

    SZ_c1_signal_kSZ = SZ.compute_combo(SZ_params_c1_kSZ, DI=True)  # Jy/sr
    c1_signals_kSZ.append(SZ_c1_signal_kSZ)

    # cluster 1 (total - tSZ + kSZ)
    SZ_params_c1_total = SZ.parameters()
    SZ_params_c1_total.betao = 0.001233586736861806
    SZ_params_c1_total.runmode = 'full'
    SZ_params_c1_total.T_order = 10
    SZ_params_c1_total.beta_order = 2

    c1_vc_idx = c1_v
    c1_Te_idx = c1_Te
    c1_tau_idx = c1_tau

    SZ_params_c1_total.muo = c1_muo
    SZ_params_c1_total.Dtau = c1_tau_idx
    SZ_params_c1_total.Te = c1_Te_idx
    SZ_params_c1_total.betac = np.abs(c1_vc_idx) * 1000 / const_c
    SZ_params_c1_total.set_x_array(0.1, 20, nfreq_samps)

    if c1_vc_idx < 0:
        SZ_params_c1_total.muc = -1
    else:
        SZ_params_c1_total.muc = 1

    # Set higher order terms to zero
    SZ_params_c1_total.means_assign_omegas(0, 0, 0)
    SZ_params_c1_total.means_assign_sigmas(0, 0, 0)
    SZ_params_c1_total.means_kappa = 0

    SZ_c1_signal_total = SZ.compute_combo(SZ_params_c1_total, DI=True)  # Jy/sr
    c1_signals_total.append(SZ_c1_signal_total)

    # cluster 2 (tSZ)
    SZ_params_c2_tSZ = SZ.parameters()
    SZ_params_c2_tSZ.betao = 0.001233586736861806
    SZ_params_c2_tSZ.runmode = 'full'
    SZ_params_c2_tSZ.T_order = 10
    SZ_params_c2_tSZ.beta_order = 2

    c2_muo = -0.5606024845891494
    c2_z = 0.071806
    c2_vc_idx = 0
    c2_Te_idx = c2_Te
    c2_tau_idx = c2_tau

    SZ_params_c2_tSZ.muo = c2_muo
    SZ_params_c2_tSZ.Dtau = c2_tau_idx
    SZ_params_c2_tSZ.Te = c2_Te_idx
    SZ_params_c2_tSZ.betac = np.abs(c2_vc_idx) * 1000 / const_c
    SZ_params_c2_tSZ.set_x_array(0.1, 20, nfreq_samps)  

    if c2_vc_idx < 0:
        SZ_params_c2_tSZ.muc = -1
    else:
        SZ_params_c2_tSZ.muc = 1

    # Set higher order terms to zero
    SZ_params_c2_tSZ.means_assign_omegas(0, 0, 0)
    SZ_params_c2_tSZ.means_assign_sigmas(0, 0, 0)
    SZ_params_c2_tSZ.means_kappa = 0
    
    SZ_c2_signal_tSZ = SZ.compute_combo(SZ_params_c2_tSZ, DI=True) 
    c2_signals_tSZ.append(SZ_c2_signal_tSZ)

    # cluster 2 (kSZ)
    SZ_params_c2_kSZ = SZ.parameters()
    SZ_params_c2_kSZ.betao = 0.001233586736861806
    SZ_params_c2_kSZ.runmode = 'full'
    SZ_params_c2_kSZ.T_order = 10
    SZ_params_c2_kSZ.beta_order = 2

    c2_vc_idx = c2_v
    c2_Te_idx = 1e-10
    c2_tau_idx = c2_tau

    SZ_params_c2_kSZ.muo = c2_muo
    SZ_params_c2_kSZ.Dtau = c2_tau_idx
    SZ_params_c2_kSZ.Te = c2_Te_idx
    SZ_params_c2_kSZ.betac = np.abs(c2_vc_idx) * 1000 / const_c
    SZ_params_c2_kSZ.set_x_array(0.1, 20, nfreq_samps)

    if c2_vc_idx < 0:
        SZ_params_c2_kSZ.muc = -1
    else:
        SZ_params_c2_kSZ.muc = 1

    # Set higher order terms to zero
    SZ_params_c2_kSZ.means_assign_omegas(0, 0, 0)
    SZ_params_c2_kSZ.means_assign_sigmas(0, 0, 0)
    SZ_params_c2_kSZ.means_kappa = 0

    SZ_c2_signal_kSZ = SZ.compute_combo(SZ_params_c2_kSZ, DI=True)
    c2_signals_kSZ.append(SZ_c2_signal_kSZ)

    # cluster 2 (total - tSZ + kSZ)
    SZ_params_c2_total = SZ.parameters()
    SZ_params_c2_total.betao = 0.001233586736861806
    SZ_params_c2_total.runmode = 'full'
    SZ_params_c2_total.T_order = 10
    SZ_params_c2_total.beta_order = 2

    c2_vc_idx = c2_v
    c2_Te_idx = c2_Te
    c2_tau_idx = c2_tau

    SZ_params_c2_total.muo = c2_muo
    SZ_params_c2_total.Dtau = c2_tau_idx
    SZ_params_c2_total.Te = c2_Te_idx
    SZ_params_c2_total.betac = np.abs(c2_vc_idx) * 1000 / const_c
    SZ_params_c2_total.set_x_array(0.1, 20, nfreq_samps)

    if c2_vc_idx < 0:
        SZ_params_c2_total.muc = -1
    else:
        SZ_params_c2_total.muc = 1

    # Set higher order terms to zero
    SZ_params_c2_total.means_assign_omegas(0, 0, 0)
    SZ_params_c2_total.means_assign_sigmas(0, 0, 0)
    SZ_params_c2_total.means_kappa = 0

    SZ_c2_signal_total = SZ.compute_combo(SZ_params_c2_total, DI=True)
    c2_signals_total.append(SZ_c2_signal_total)

    # filament (tSZ)
    SZ_params_fil_tSZ = SZ.parameters()
    SZ_params_fil_tSZ.betao = 0.001233586736861806
    SZ_params_fil_tSZ.runmode = 'full'
    SZ_params_fil_tSZ.T_order = 10
    SZ_params_fil_tSZ.beta_order = 2

    fil_muo = -0.5576451231598584
    fil_z = 0.072735
    fil_vc_idx = 0
    fil_Te_idx = fil_Te
    fil_tau_idx = fil_tau

    SZ_params_fil_tSZ.muo = fil_muo
    SZ_params_fil_tSZ.Dtau = fil_tau_idx
    SZ_params_fil_tSZ.Te = fil_Te_idx
    SZ_params_fil_tSZ.betac = np.abs(fil_vc_idx) * 1000 / const_c
    SZ_params_fil_tSZ.set_x_array(0.1, 20, nfreq_samps)

    if fil_vc_idx < 0:
        SZ_params_fil_tSZ.muc = -1
    else:
        SZ_params_fil_tSZ.muc = 1
    
    # Set higher order terms to zero
    SZ_params_fil_tSZ.means_assign_omegas(0, 0, 0)
    SZ_params_fil_tSZ.means_assign_sigmas(0, 0, 0)
    SZ_params_fil_tSZ.means_kappa = 0

    SZ_fil_signal_tSZ = SZ.compute_combo(SZ_params_fil_tSZ, DI=True) 
    fil_signals_tSZ.append(SZ_fil_signal_tSZ)

    # filament (kSZ)
    SZ_params_fil_kSZ = SZ.parameters()
    SZ_params_fil_kSZ.betao = 0.001233586736861806
    SZ_params_fil_kSZ.runmode = 'full'
    SZ_params_fil_kSZ.T_order = 10
    SZ_params_fil_kSZ.beta_order = 2

    fil_vc_idx = fil_v
    fil_Te_idx = 1e-10
    fil_tau_idx = fil_tau

    SZ_params_fil_kSZ.muo = fil_muo
    SZ_params_fil_kSZ.Dtau = fil_tau_idx
    SZ_params_fil_kSZ.Te = fil_Te_idx
    SZ_params_fil_kSZ.betac = np.abs(fil_vc_idx) * 1000 / const_c
    SZ_params_fil_kSZ.set_x_array(0.1, 20, nfreq_samps)

    if fil_vc_idx < 0:
        SZ_params_fil_kSZ.muc = -1
    else:
        SZ_params_fil_kSZ.muc = 1
    
    # Set higher order terms to zero
    SZ_params_fil_kSZ.means_assign_omegas(0, 0, 0)
    SZ_params_fil_kSZ.means_assign_sigmas(0, 0, 0)
    SZ_params_fil_kSZ.means_kappa = 0

    SZ_fil_signal_kSZ = SZ.compute_combo(SZ_params_fil_kSZ, DI=True)
    fil_signals_kSZ.append(SZ_fil_signal_kSZ)

    # filament (total - tSZ + kSZ)
    SZ_params_fil_total = SZ.parameters()
    SZ_params_fil_total.betao = 0.001233586736861806
    SZ_params_fil_total.runmode = 'full'
    SZ_params_fil_total.T_order = 10
    SZ_params_fil_total.beta_order = 2

    fil_vc_idx = fil_v
    fil_Te_idx = fil_Te
    fil_tau_idx = fil_tau

    SZ_params_fil_total.muo = fil_muo
    SZ_params_fil_total.Dtau = fil_tau_idx
    SZ_params_fil_total.Te = fil_Te_idx
    SZ_params_fil_total.betac = np.abs(fil_vc_idx) * 1000 / const_c
    SZ_params_fil_total.set_x_array(0.1, 20, nfreq_samps)

    if fil_vc_idx < 0:
        SZ_params_fil_total.muc = -1
    else:
        SZ_params_fil_total.muc = 1
    
    # Set higher order terms to zero
    SZ_params_fil_total.means_assign_omegas(0, 0, 0)
    SZ_params_fil_total.means_assign_sigmas(0, 0, 0)
    SZ_params_fil_total.means_kappa = 0

    SZ_fil_signal_total = SZ.compute_combo(SZ_params_fil_total, DI=True)
    fil_signals_total.append(SZ_fil_signal_total)


In [ ]:
import matplotlib.ticker as ticker

# Process tSZ signals
c1_signals_tSZ_stacked = np.stack(c1_signals_tSZ)
c1_signals_tSZ_percentile_16 = np.percentile(c1_signals_tSZ_stacked, 16, axis=0)
c1_signals_tSZ_percentile_84 = np.percentile(c1_signals_tSZ_stacked, 84, axis=0)
c1_signals_tSZ_mean = np.median(c1_signals_tSZ_stacked, axis=0)

c2_signals_tSZ_stacked = np.stack(c2_signals_tSZ)
c2_signals_tSZ_percentile_16 = np.percentile(c2_signals_tSZ_stacked, 16, axis=0)
c2_signals_tSZ_percentile_84 = np.percentile(c2_signals_tSZ_stacked, 84, axis=0)
c2_signals_tSZ_mean = np.median(c2_signals_tSZ_stacked, axis=0)

fil_signals_tSZ_stacked = np.stack(fil_signals_tSZ)
fil_signals_tSZ_percentile_16 = np.percentile(fil_signals_tSZ_stacked, 16, axis=0)
fil_signals_tSZ_percentile_84 = np.percentile(fil_signals_tSZ_stacked, 84, axis=0)
fil_signals_tSZ_mean = np.median(fil_signals_tSZ_stacked, axis=0)

# Process kSZ signals
c1_signals_kSZ_stacked = np.stack(c1_signals_kSZ)
c1_signals_kSZ_percentile_16 = np.percentile(c1_signals_kSZ_stacked, 16, axis=0)
c1_signals_kSZ_percentile_84 = np.percentile(c1_signals_kSZ_stacked, 84, axis=0)
c1_signals_kSZ_mean = np.median(c1_signals_kSZ_stacked, axis=0)

c2_signals_kSZ_stacked = np.stack(c2_signals_kSZ)
c2_signals_kSZ_percentile_16 = np.percentile(c2_signals_kSZ_stacked, 16, axis=0)
c2_signals_kSZ_percentile_84 = np.percentile(c2_signals_kSZ_stacked, 84, axis=0)
c2_signals_kSZ_mean = np.median(c2_signals_kSZ_stacked, axis=0)

fil_signals_kSZ_stacked = np.stack(fil_signals_kSZ)
fil_signals_kSZ_percentile_16 = np.percentile(fil_signals_kSZ_stacked, 16, axis=0)
fil_signals_kSZ_percentile_84 = np.percentile(fil_signals_kSZ_stacked, 84, axis=0)
fil_signals_kSZ_mean = np.median(fil_signals_kSZ_stacked, axis=0)

# Process total signals (tSZ + kSZ)
c1_signals_total_stacked = np.stack(c1_signals_total)
c1_signals_total_percentile_16 = np.percentile(c1_signals_total_stacked, 16, axis=0)
c1_signals_total_percentile_84 = np.percentile(c1_signals_total_stacked, 84, axis=0)
c1_signals_total_mean = np.median(c1_signals_total_stacked, axis=0)

c2_signals_total_stacked = np.stack(c2_signals_total)
c2_signals_total_percentile_16 = np.percentile(c2_signals_total_stacked, 16, axis=0)
c2_signals_total_percentile_84 = np.percentile(c2_signals_total_stacked, 84, axis=0)
c2_signals_total_mean = np.median(c2_signals_total_stacked, axis=0)

fil_signals_total_stacked = np.stack(fil_signals_total)
fil_signals_total_percentile_16 = np.percentile(fil_signals_total_stacked, 16, axis=0)
fil_signals_total_percentile_84 = np.percentile(fil_signals_total_stacked, 84, axis=0)
fil_signals_total_mean = np.median(fil_signals_total_stacked, axis=0)

plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=32)

fig = plt.figure(figsize=(16, 6))
ax1 = fig.add_subplot(111)

# Plot C1 signals (Abell 401) - blue for cluster 1
ax1.plot(SZ_params_c1_tSZ.nucmb, c1_signals_tSZ_mean, label='Abell 401 (tSZ)', color='blue', linestyle='-', lw=2)
ax1.fill_between(SZ_params_c1_tSZ.nucmb, 
                 c1_signals_tSZ_percentile_16, 
                 c1_signals_tSZ_percentile_84,
                 color='blue', alpha=0.2)

ax1.plot(SZ_params_c1_kSZ.nucmb, c1_signals_kSZ_mean, label='Abell 401 (kSZ)', color='lightblue', linestyle='--', lw=2)
ax1.fill_between(SZ_params_c1_kSZ.nucmb,
                 c1_signals_kSZ_percentile_16,
                 c1_signals_kSZ_percentile_84,
                 color='lightblue', alpha=0.2)

ax1.plot(SZ_params_c1_total.nucmb, c1_signals_total_mean, label='Abell 401 (total)', color='darkblue', linestyle=':', lw=3)
ax1.fill_between(SZ_params_c1_total.nucmb,
                 c1_signals_total_percentile_16,
                 c1_signals_total_percentile_84,
                 color='darkblue', alpha=0.2)

# Plot C2 signals (Abell 399) - red for cluster 2
ax1.plot(SZ_params_c2_tSZ.nucmb, c2_signals_tSZ_mean, label='Abell 399 (tSZ)', color='red', linestyle='-', lw=2)
ax1.fill_between(SZ_params_c2_tSZ.nucmb,
                 c2_signals_tSZ_percentile_16,
                 c2_signals_tSZ_percentile_84,
                 color='red', alpha=0.2)

ax1.plot(SZ_params_c2_kSZ.nucmb, c2_signals_kSZ_mean, label='Abell 399 (kSZ)', color='lightcoral', linestyle='--', lw=2)
ax1.fill_between(SZ_params_c2_kSZ.nucmb,
                 c2_signals_kSZ_percentile_16,
                 c2_signals_kSZ_percentile_84,
                 color='lightcoral', alpha=0.2)

ax1.plot(SZ_params_c2_total.nucmb, c2_signals_total_mean, label='Abell 399 (total)', color='darkred', linestyle=':', lw=3)
ax1.fill_between(SZ_params_c2_total.nucmb,
                 c2_signals_total_percentile_16,
                 c2_signals_total_percentile_84,
                 color='darkred', alpha=0.2)

# Plot Filament signals - green for filament
ax1.plot(SZ_params_fil_tSZ.nucmb, fil_signals_tSZ_mean, label='Filament (tSZ)', color='green', linestyle='-', lw=2)
ax1.fill_between(SZ_params_fil_tSZ.nucmb,
                 fil_signals_tSZ_percentile_16,
                 fil_signals_tSZ_percentile_84,
                 color='green', alpha=0.2)

ax1.plot(SZ_params_fil_kSZ.nucmb, fil_signals_kSZ_mean, label='Filament (kSZ)', color='lightgreen', linestyle='--', lw=2)
ax1.fill_between(SZ_params_fil_kSZ.nucmb,
                 fil_signals_kSZ_percentile_16,
                 fil_signals_kSZ_percentile_84,
                 color='lightgreen', alpha=0.2)

ax1.plot(SZ_params_fil_total.nucmb, fil_signals_total_mean, label='Filament (total)', color='darkgreen', linestyle=':', lw=3)
ax1.fill_between(SZ_params_fil_total.nucmb,
                 fil_signals_total_percentile_16,
                 fil_signals_total_percentile_84,
                 color='darkgreen', alpha=0.2)

ax1.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax1.set_xlabel(r'Dimensionless frequency, $x= h\nu/k_{\rm B}T_{\rm CMB}$')
ax1.set_ylabel('$\Delta I$ [MJy/Sr]')
ax1.legend(fontsize=20, frameon=0, loc='upper right', ncol=3)

plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Frequency (GHz)')
plt.xticks(np.arange(0, 1200, 100))
#plt.savefig("../plots/sz_spectrum_tSZ_kSZ_total.pdf", bbox_inches='tight', dpi=400)

# vertical line at freq = 218
plt.axvline(220, color='black', linestyle='--')

plt.show()


In [ ]:
# Get the log probabilities for all samples with the same thinning as samples
log_probs = sampler.get_log_prob(discard=50000, flat=True, thin=1000)

# Find the index of the maximum likelihood
max_like_idx = np.argmax(log_probs)

# Get the maximum likelihood parameters
max_like_params = samples[max_like_idx]

print(f"Maximum likelihood index: {max_like_idx}")
print(f"Maximum log probability: {log_probs[max_like_idx]:.6f}")
print(f"Number of samples: {len(samples)}")
print(f"Number of log probabilities: {len(log_probs)}")
print("\nMaximum likelihood parameters:")
for i, (label, param) in enumerate(zip(labels, max_like_params)):
    print(f"{label}: {param:.6f}")

In [ ]:
labels = [r'$RA_{\rm A401}$', 
            r'$DEC_{\rm A401}$', 
            r'$\beta_{\rm A401}$', 
            r'$r_{\rm c, A401}$ [$^{\prime}$]', 
            r'$e_{\rm A401}$', 
            r'$\theta_{\rm A401}$', 
            r'$\tau_{\rm A401}$', 
            r'$T_{\rm e, A401}$', 
            r'$A_{\rm D, A401}$',
            r"$v_{r, A401}$",
            
            r'$RA_{\rm A399}$', 
            r'$DEC_{\rm A399}$', 
            r'$\beta_{\rm A399}$', 
            r'$r_{\rm c, A399}$ [$^{\prime}$]', 
            r'$e_{\rm A399}$', 
            r'$\theta_{\rm A399}$', 
            r'$\tau_{\rm A399}$', 
            r'$T_{\rm e, A399}$', 
            r'$A_{\rm D, A399}$',
            r"$v_{r, A399}$",

            r"$RA_{\rm fil}$",
            r"$DEC_{\rm fil}$",
            r"$L_{\rm fil}$",
            r"$W_{\rm fil}$",
            r'$\tau_{\rm fil}$',
            r'$T_{\rm e, fil}$',
            r'$A_{\rm D, fil}$',
            
            r"$v_{r, avg}$"]

const_c = 299792458.0 # m / s
const_k_B = 1.380649e-23 # J / K
const_h = 6.626070149999999e-25 # J / GHz

c1_signals = []
c2_signals = []
fil_signals = []

dust_c1_signals = []
dust_c2_signals = []
dust_fil_signals = []

nfreq_samps = 500

for sample in samples:
    c1_tau = sample[6]
    c1_Te = sample[7]
    c1_A_D = sample[8]  
    c1_v = sample[9]

    c2_tau = sample[16]
    c2_Te = sample[17]
    c2_A_D = sample[18]
    c2_v = sample[19]

    fil_tau = sample[24]
    fil_Te = sample[25]
    fil_A_D = sample[26]
    fil_v = sample[27]

    SZ_params_c1 = SZ.parameters()
    SZ_params_c1.betao = 0.001233586736861806
    SZ_params_c1.runmode = 'full'
    SZ_params_c1.T_order = 10
    SZ_params_c1.beta_order = 2

    SZ_params_c2 = SZ.parameters()
    SZ_params_c2.betao = 0.001233586736861806
    SZ_params_c2.runmode = 'full'
    SZ_params_c2.T_order = 10
    SZ_params_c2.beta_order = 2

    SZ_params_fil = SZ.parameters()
    SZ_params_fil.betao = 0.001233586736861806
    SZ_params_fil.runmode = 'full'
    SZ_params_fil.T_order = 10
    SZ_params_fil.beta_order = 2

    # Dust parameters
    freq_ref = 545 
    T_D = 20 
    beta_D = 1.5

    # cluster 1
    c1_muo = -0.5566122489553409
    c1_z = 0.073664
    c1_vc_idx = c1_v
    c1_Te_idx = c1_Te
    c1_tau_idx = c1_tau

    SZ_params_c1.muo = c1_muo
    SZ_params_c1.Dtau = c1_tau_idx
    SZ_params_c1.Te = c1_Te_idx
    SZ_params_c1.betac = np.abs(c1_vc_idx) * 1000 / const_c
    SZ_params_c1.set_x_array(0.1, 20, nfreq_samps)  

    if c1_vc_idx < 0:
        SZ_params_c1.muc = -1
    else:
        SZ_params_c1.muc = 1

    # Set higher order terms to zero
    SZ_params_c1.means_assign_omegas(0, 0, 0)
    SZ_params_c1.means_assign_sigmas(0, 0, 0)
    SZ_params_c1.means_kappa = 0 

    SZ_c1_signal = SZ.compute_combo(SZ_params_c1, DI=True)  # Jy/sr
    c1_signals.append(SZ_c1_signal)

    c1_AD_idx = c1_A_D
    x_dust = (const_h * SZ_params_c1.nucmb * (1 + c1_z)) / (const_k_B * T_D)     
    x_ref = (const_h * freq_ref) / (const_k_B * T_D)
    term_ref = (np.exp(x_ref) - 1)
    term_dust = (np.exp(x_dust) - 1)
    term_power = (SZ_params_c1.nucmb * (1 + c1_z) / freq_ref)**(beta_D + 3)
    I_dust = c1_AD_idx * term_power * (term_ref / term_dust) / 10**6
    dust_c1_signals.append(I_dust)

    # cluster 2
    c2_muo = -0.5606024845891494
    c2_z = 0.071806
    c2_vc_idx = c2_v
    c2_Te_idx = c2_Te
    c2_tau_idx = c2_tau

    SZ_params_c2.muo = c2_muo
    SZ_params_c2.Dtau = c2_tau_idx
    SZ_params_c2.Te = c2_Te_idx
    SZ_params_c2.betac = np.abs(c2_vc_idx) * 1000 / const_c
    SZ_params_c2.set_x_array(0.1, 20, nfreq_samps)  

    if c2_vc_idx < 0:
        SZ_params_c2.muc = -1
    else:
        SZ_params_c2.muc = 1

    # Set higher order terms to zero
    SZ_params_c2.means_assign_omegas(0, 0, 0)
    SZ_params_c2.means_assign_sigmas(0, 0, 0)
    SZ_params_c2.means_kappa = 0
    
    SZ_c2_signal = SZ.compute_combo(SZ_params_c2, DI=True) 
    c2_signals.append(SZ_c2_signal)

    c2_AD_idx = c2_A_D
    x_dust = (const_h * SZ_params_c2.nucmb * (1 + c2_z)) / (const_k_B * T_D)
    x_ref = (const_h * freq_ref) / (const_k_B * T_D)
    term_ref = (np.exp(x_ref) - 1)
    term_dust = (np.exp(x_dust) - 1)
    term_power = (SZ_params_c2.nucmb * (1 + c2_z) / freq_ref)**(beta_D + 3)
    I_dust = c2_AD_idx * term_power * (term_ref / term_dust) / 10**6
    dust_c2_signals.append(I_dust)

    # filament
    fil_muo = -0.5576451231598584
    fil_z = 0.072735
    fil_vc_idx = fil_v
    fil_Te_idx = fil_Te
    fil_tau_idx = fil_tau

    SZ_params_fil.muo = fil_muo
    SZ_params_fil.Dtau = fil_tau_idx
    SZ_params_fil.Te = fil_Te_idx
    SZ_params_fil.betac = np.abs(fil_vc_idx) * 1000 / const_c
    SZ_params_fil.set_x_array(0.1, 20, nfreq_samps)

    if fil_vc_idx < 0:
        SZ_params_fil.muc = -1
    else:
        SZ_params_fil.muc = 1
    
    # Set higher order terms to zero
    SZ_params_fil.means_assign_omegas(0, 0, 0)
    SZ_params_fil.means_assign_sigmas(0, 0, 0)
    SZ_params_fil.means_kappa = 0

    SZ_fil_signal = SZ.compute_combo(SZ_params_fil, DI=True) 
    fil_signals.append(SZ_fil_signal)

    fil_AD_idx = fil_A_D
    x_dust = (const_h * SZ_params_fil.nucmb * (1 + fil_z)) / (const_k_B * T_D)
    x_ref = (const_h * freq_ref) / (const_k_B * T_D)
    term_ref = (np.exp(x_ref) - 1)
    term_dust = (np.exp(x_dust) - 1)
    term_power = (SZ_params_fil.nucmb * (1 + fil_z) / freq_ref)**(beta_D + 3)
    I_dust = fil_AD_idx * term_power * (term_ref / term_dust) / 10**6
    dust_fil_signals.append(I_dust)


In [ ]:
import matplotlib.ticker as ticker

# Process c1 signals (SZ only)
c1_signals_stacked = np.stack(c1_signals)
c1_signals_percentile_16 = np.percentile(c1_signals_stacked, 16, axis=0)
c1_signals_percentile_84 = np.percentile(c1_signals_stacked, 84, axis=0)
c1_signals_mean =np.median(c1_signals_stacked, axis=0)

# Process c2 signals (SZ only)
c2_signals_stacked = np.stack(c2_signals)
c2_signals_percentile_16 = np.percentile(c2_signals_stacked, 16, axis=0)
c2_signals_percentile_84 = np.percentile(c2_signals_stacked, 84, axis=0)
c2_signals_mean =np.median(c2_signals_stacked, axis=0)

# Process filament signals (SZ only)
fil_signals_stacked = np.stack(fil_signals)
fil_signals_percentile_16 = np.percentile(fil_signals_stacked, 16, axis=0)
fil_signals_percentile_84 = np.percentile(fil_signals_stacked, 84, axis=0)
fil_signals_mean =np.median(fil_signals_stacked, axis=0)

plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=32)

fig = plt.figure(figsize=(11, 6))
ax1 = fig.add_subplot(111)

# Plot C1 SZ signals (Abell 401)
ax1.plot(SZ_params_c1.nucmb, c1_signals_mean, label='Abell 401', color='blue', lw=1)
ax1.fill_between(SZ_params_c1.nucmb, 
                 c1_signals_percentile_16, 
                 c1_signals_percentile_84,
                 color='blue', alpha=0.2)

# Plot C2 SZ signals (Abell 399)
ax1.plot(SZ_params_c2.nucmb, c2_signals_mean, label='Abell 399', color='red', lw=1)
ax1.fill_between(SZ_params_c2.nucmb,
                 c2_signals_percentile_16,
                 c2_signals_percentile_84,
                 color='red', alpha=0.2)

# Plot Filament SZ signals
ax1.plot(SZ_params_fil.nucmb, fil_signals_mean, label='Filament', color='green', lw=1)
ax1.fill_between(SZ_params_fil.nucmb,
                 fil_signals_percentile_16,
                 fil_signals_percentile_84,
                 color='green', alpha=0.2)

ax1.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax1.set_xlabel(r'Dimensionless frequency, $x= h\nu/k_{\rm B}T_{\rm CMB}$')
ax1.set_ylabel('$\Delta I$ [MJy/Sr]')
ax1.legend(fontsize=30, frameon=0, loc='upper right')

plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Frequency (GHz)')
plt.xticks(np.arange(0, 1200, 200))
plt.savefig("../plots/sz_spectrum_ind.pdf", bbox_inches='tight', dpi=400)
plt.show()


In [ ]:
import matplotlib.ticker as ticker

dust_c1_signals_stacked = np.stack(dust_c1_signals)
dust_c1_signals_percentile_16 = np.percentile(dust_c1_signals_stacked, 16, axis=0)
dust_c1_signals_percentile_84 = np.percentile(dust_c1_signals_stacked, 84, axis=0)
dust_c1_signals_mean = np.median(dust_c1_signals_stacked, axis=0)

dust_c2_signals_stacked = np.stack(dust_c2_signals)
dust_c2_signals_percentile_16 = np.percentile(dust_c2_signals_stacked, 16, axis=0)
dust_c2_signals_percentile_84 = np.percentile(dust_c2_signals_stacked, 84, axis=0)
dust_c2_signals_mean = np.median(dust_c2_signals_stacked, axis=0)

dust_fil_signals_stacked = np.stack(dust_fil_signals)
dust_fil_signals_percentile_16 = np.percentile(dust_fil_signals_stacked, 16, axis=0)
dust_fil_signals_percentile_84 = np.percentile(dust_fil_signals_stacked, 84, axis=0)
dust_fil_signals_mean = np.median(dust_fil_signals_stacked, axis=0)

plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=32)

fig = plt.figure(figsize=(11, 6))
ax1 = fig.add_subplot(111)

# Dust C1 signals
ax1.plot(SZ_params_c1.nucmb, dust_c1_signals_mean, label='Abell 401', color='blue', linestyle='-', lw=1)
ax1.fill_between(SZ_params_c1.nucmb,
                 dust_c1_signals_percentile_16,
                 dust_c1_signals_percentile_84,
                 color='blue', alpha=0.2)

# Dust C2 signals
ax1.plot(SZ_params_c2.nucmb, dust_c2_signals_mean, label='Abell 399', color='red', linestyle='-', lw=1)
ax1.fill_between(SZ_params_c2.nucmb,
                 dust_c2_signals_percentile_16,
                 dust_c2_signals_percentile_84,
                 color='red', alpha=0.2)

# Dust Filament signals
ax1.plot(SZ_params_fil.nucmb, dust_fil_signals_mean, label='Filament', color='green', linestyle='-', lw=1)
ax1.fill_between(SZ_params_fil.nucmb,
                 dust_fil_signals_percentile_16,
                 dust_fil_signals_percentile_84,
                 color='green', alpha=0.2)

ax1.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax1.set_xlabel(r'Dimensionless frequency, $x= h\nu/k_{\rm B}T_{\rm CMB}$')
ax1.set_ylabel('$\Delta I$ [MJy/Sr]')
ax1.legend(fontsize=30, frameon=0, loc='upper left')

plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Frequency (GHz)')
plt.xticks(np.arange(0, 1200, 200))

plt.savefig(f"../plots/dust_spectrum_ind.pdf", bbox_inches='tight', dpi=400)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from astropy.coordinates import SkyCoord
from astropy import units as u
from pixell import enmap

# Define cluster coordinates (from the fit results)
ra_c2_fit = 44.9433173  # A401 RA
dec_c2_fit = 13.9563979  # A401 Dec
ra_c1_fit = 44.3304539  # A399 RA  
dec_c1_fit = 12.7909530  # A399 Dec

# Create line connecting the two clusters
n_points = 10000
ra_line = np.linspace(ra_c1_fit, ra_c2_fit, n_points)
dec_line = np.linspace(dec_c1_fit, dec_c2_fit, n_points)

# Load one of the saved models to show the line on the map
model_file = "/home/gill/research/ACT/paper/models/models_july12/98_model.fits"
model_map = enmap.read_map(model_file)

# Create the plot
plt.figure(figsize=(10, 8))
ax = plt.subplot(projection=model_map.wcs)

# Display the model map
im = ax.imshow(model_map, origin='lower', cmap='planck', 
                                                   vmin=-80, vmax=80, interpolation='none')

# Plot the line connecting the clusters
ax.plot(ra_line, dec_line, 'r-', linewidth=3, 
                                transform=ax.get_transform('world'), label='Connection line')

# Mark the cluster positions
ax.plot(ra_c1_fit, dec_c1_fit, 'bo', markersize=10, 
                                transform=ax.get_transform('world'), label='A401')
ax.plot(ra_c2_fit, dec_c2_fit, 'go', markersize=10, 
                                transform=ax.get_transform('world'), label='A399')

# Set up coordinates
ax.coords[0].set_axislabel('Right Ascension')
ax.coords[1].set_axislabel('Declination')
ax.coords[0].set_major_formatter('d')
ax.coords[1].set_major_formatter('d')
ax.invert_xaxis()

# Add colorbar and legend
plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.1, 
                                                 label='SZ Signal (kJy/sr)', fraction=0.046)
plt.legend()
plt.title('SZ Model at 98 GHz with Line Connecting A401 and A399')

plt.tight_layout()
plt.savefig("../plots/line_connection_map.pdf", bbox_inches='tight', dpi=300)
plt.show()

# Print line information
distance_deg = np.sqrt((ra_c2_fit - ra_c1_fit)**2 + (dec_c2_fit - dec_c1_fit)**2)
distance_arcmin = distance_deg * 60
print(f"Distance between clusters: {distance_arcmin:.2f} arcmin")
print(f"A401 position: RA={ra_c1_fit:.6f}, Dec={dec_c1_fit:.6f}")
print(f"A399 position: RA={ra_c2_fit:.6f}, Dec={dec_c2_fit:.6f}")

# Extract signal along the line - FIXED coordinate conversion with interpolation
from scipy.interpolate import RectBivariateSpline

# Create coordinate arrays for interpolation
y_coords = np.arange(model_map.shape[0])
x_coords = np.arange(model_map.shape[1])

# Create interpolation function
interpolator = RectBivariateSpline(y_coords, x_coords, model_map, kx=1, ky=1)

signal_along_line_98 = []
for ra, dec in zip(ra_line, dec_line):
        # Convert world coordinates to pixel coordinates
        pix_coords = model_map.sky2pix([dec * np.pi/180, ra * np.pi/180])  # Convert to radians
        
        # Use interpolation instead of integer indexing
        # Note: RectBivariateSpline expects (y, x) order
        if (0 <= pix_coords[0] < model_map.shape[0] and 
            0 <= pix_coords[1] < model_map.shape[1]):
            pix_val = interpolator(pix_coords[0], pix_coords[1])[0, 0]
        else:
            pix_val = 0.0  # Handle points outside map bounds
        
        signal_along_line_98.append(pix_val)

# Create distance array (in arcmin)
distance = np.linspace(0, distance_arcmin, n_points)

# Plot the signal
plt.figure(figsize=(12, 8))
plt.plot(distance, np.array(signal_along_line_98), 'b-', linewidth=2, label='98 GHz')
plt.xlabel('Distance along line (arcmin)', fontsize=12)
plt.ylabel('SZ Signal (Jy/sr)', fontsize=12)
plt.title('Signal Along Line Connecting A401 and A399 at 98 GHz', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(0, color='k', linestyle='--', alpha=0.5)
plt.legend(fontsize=12)

# Add vertical lines to mark cluster positions
plt.axvline(0, color='blue', linestyle=':', alpha=0.7, label='A401 position')
plt.axvline(distance_arcmin, color='green', linestyle=':', alpha=0.7, label='A399 position')

plt.tight_layout()
plt.savefig("../plots/signal_along_line_98GHz.pdf", bbox_inches='tight', dpi=300)
plt.show()

# Print some statistics
print(f"Maximum signal along line: {np.max(signal_along_line_98):.2f} Jy/sr")
print(f"Minimum signal along line: {np.min(signal_along_line_98):.2f} Jy/sr")
print(f"Mean signal along line: {np.mean(signal_along_line_98):.2f} Jy/sr")


In [ ]:
stacked = np.stack(c1_signals)  # shape: (10, 1000)

# Compute percentiles along the 0th axis (i.e., across the arrays)
percentile_16 = np.percentile(stacked, 16, axis=0)
percentile_84 = np.percentile(stacked, 84, axis=0)

# Compute the mean
mean = np.mean(stacked, axis=0)
# Compute the standard deviation
std_dev = np.std(stacked, axis=0)

print(percentile_16)
print(percentile_84)

print(mean)
print(std_dev)

In [ ]:
freq_ref = 545 
T_D = 20 
beta_D = 1.5
const_c = 299792458.0 # m / s
const_k_B = 1.380649e-23 # J / K
const_h = 6.626070149999999e-25 # J / GHz
z = 0.073664

freq_array = np.arange(30, 590, 1)

x_dust = (const_h * freq_array * (1 + z) ) / (const_k_B * T_D)     
x_ref = (const_h * freq_ref) / (const_k_B * T_D)

term_ref = (np.exp(x_ref) - 1)
term_dust = (np.exp(x_dust) - 1)

term_power = (freq_array * (1+z) / freq_ref)**(beta_D + 3)

I_dust = 500506.948182 * term_power * (term_ref / term_dust) 

I_dust

In [ ]:
import matplotlib.ticker as ticker

plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=32)

fig = plt.figure(figsize=(13, 7))
ax1 = fig.add_subplot(111)
ax2 = ax1.twiny()

ax2.plot(SZParams.nucmb, np.zeros(SZParams.gridpoints),'k',alpha=0)  #Twinning the nu values to the x values
ax1.plot(SZParams.xcmb, DI, label='tSZ')

ax1.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax2.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax1.set_xlabel(r'Dimensionless frequency, $x= h\nu/k_{\rm B}T_{\rm CMB}$')
ax2.set_xlabel('Frequency [GHz]')
ax1.set_ylabel('Spectral distortion, $\Delta I$ [MJy/Sr]')
# ax1.set_ylim(-0.4,0.6)

for axs in [ax1,ax2]:
    for axis in ['top','bottom','left','right']:
        axs.spines[axis].set_linewidth(1.2)
    axs.tick_params(which='both', width=1.2)
    axs.tick_params(which='major', length=7)
    axs.tick_params(which='minor', length=4)